# Built-in Tools & Toolkits

You don't have to build every tool yourself. LangChain ships **ready-made tools** (web search, Python
execution, Wikipedia, etc.) and **toolkits** (bundles of related tools for a system like a database or
Gmail). Reuse before you reinvent.

---

## 1. Simple Definition

> **Kid version:** You don't need to build every tool from scratch. But the store already sells lots
> of gadgets — a web-searcher, a calculator, a Wikipedia-reader. A **toolkit** is a whole boxed set of
> matching gadgets for one job (like a full "database repair kit"). Grab those instead of building
> from scratch.

**Professional definition:**
- A **built-in tool** is a pre-implemented `BaseTool` provided by LangChain or its integration
  packages (e.g., search, REPL).
- A **toolkit** is a class that groups related tools for a specific system, exposing them via
  `.get_tools()`.

```python
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
search.invoke("latest langchain release")   # returns search results as a string
```

---

## 2. Why Do They Exist?

**The problem:** Common capabilities (web search, Wikipedia, running Python, querying SQL) are needed
by *everyone*. Rewriting them — with correct schemas, error handling, and API plumbing — wastes time
and invites bugs.

### Before (roll your own web search)

```python
@tool
def web_search(query: str) -> str:
    """Search the web."""
    # sign up for an API, handle auth, pagination, rate limits, parsing... 😩
    ...
```

### After (use a built-in)

```python
from langchain_community.tools.tavily_search import TavilySearchResults
search = TavilySearchResults(max_results=3)   # done — battle-tested
```

Built-ins are already `BaseTool`s, so they drop straight into `bind_tools([...])` and the loop.

---

## 3. Real-Life Analogy

**IKEA vs. building furniture from raw timber** 🪑. A **built-in tool** is a ready-made chair. A
**toolkit** is a full flat-pack *set* (table + 4 matching chairs) designed to work together for one
room (one system, like "your SQL database"). You *can* carve your own, but usually you shouldn't.

---

## 4. Where They Fit in LangChain Architecture

```
   langchain_community / integration packages
        │
        ├── built-in tools   (DuckDuckGoSearchRun, WikipediaQueryRun, PythonREPLTool, ...)
        │        └── each IS a BaseTool → drop into bind_tools([...])
        │
        └── toolkits         (SQLDatabaseToolkit, GmailToolkit, FileManagementToolkit, ...)
                 └── .get_tools() → returns a LIST of related BaseTools
```

Everything ends up as `BaseTool`s feeding into `bind_tools` and the loop.

> 📦 Many built-ins live in `langchain_community` or provider packages (e.g. `langchain_tavily`).
> Install the relevant package; APIs move around across versions, so check imports for your version.

---

## 5. Commonly used built-in tools

### `Web search (Tavily / DuckDuckGo / SerpAPI)`

**Definition:** Query a web-search API and return results as text.

**Why it exists:** Give the model live, post-training-cutoff information.

**When developers use it:** Research assistants, "current events" Q&A, RAG-with-web.

In [1]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool = DuckDuckGoSearchRun()
result = search_tool.invoke("Tell me about CJP protest")
print(result)

C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_11668\575319334.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


A series of demonstrations took place in Jantar Mantar, New Delhi, India, between 6 June and 25 July 2026. They were initiated by the Cockroach Janta Party (CJP), an Indian youth-based satirical political movement, along with left-wing student organisations [16][17] including the All India Students Federation, All India Students Association, and Students' Federation of India. [18][19] Held ... Volunteers like an electrician, thelawala, doctor and students keep the CJP Jantar Mantar protest running as police ID checks scrutinize food and water supplies. Abhijeet Dipke Jantar Mantar protest: Cockroach Janta Party (CJP) launched its protest at Delhi's Jantar Mantar on Saturday, demanding Union Education Minister Dharmendra Pradhan's resignation over the NEET 2026 paper leak and alleged CBSE OSM irregularities. Founder Abhijeet Dipke arrived from Boston to lead the demonstration as protesters raised anti-government slogans. The CJP has spearheaded the recent protests over allegations that 

In [2]:
print("=" * 40)
print("🛠️  Tool Information")
print("=" * 40)
print(f"Name        : {search_tool.name}")
print(f"Description : {search_tool.description}")
print(f"Arguments   : {search_tool.args}")
print("=" * 40)

🛠️  Tool Information
Name        : duckduckgo_search
Description : A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
Arguments   : {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


### `Python REPL`

**Definition:** Execute Python code the model writes and return the output.

**Why it exists:** Exact math, data manipulation, plotting — things the model shouldn't *guess*.

**When developers use it:** Data analysis agents, precise calculation.

In [3]:
from langchain_experimental.tools import PythonREPLTool
py = PythonREPLTool()
result = py.invoke("print(sum(range(10)))")
print(result)

C:\Users\Mr. Sachin Kapoor\AppData\Local\Temp\ipykernel_11668\1551849229.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonREPLTool
Python REPL can execute arbitrary code. Use with caution.


45



In [4]:
print("=" * 40)
print("🛠️  Tool Information")
print("=" * 40)
print(f"Name        : {py.name}")
print(f"Description : {py.description}")
print(f"Arguments   : {py.args}")
print("=" * 40)

🛠️  Tool Information
Name        : Python_REPL
Description : A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.
Arguments   : {'query': {'title': 'Query', 'type': 'string'}}


### `Wikipedia`

**Definition:** Look up and summarize Wikipedia articles.

**Why it exists:** Reliable encyclopedic facts on demand.

In [5]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Create the Wikipedia API wrapper
api_wrapper = WikipediaAPIWrapper(top_k_results=1,
                                  doc_content_chars_max=500)

# Create the tool
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper)

# Query Wikipedia
result = wikipedia.invoke("who is the president of the United States?")

print(result)

Page: President of the United States
Summary: The president of the United States (POTUS) is the head of state and head of government of the United States. The president directs the executive branch of the federal government and is the commander-in-chief of the United States Armed Forces.
The power of the presidency has grown since the first president, George Washington, took office in 1789. While presidential power has ebbed and flowed over time, the presidency has played an increasing role in A


In [6]:
print("=" * 40)
print("🛠️  Tool Information")
print("=" * 40)
print(f"Name        : {wikipedia.name}")
print(f"Description : {wikipedia.description}")
print(f"Arguments   : {wikipedia.args}")
print("=" * 40)

🛠️  Tool Information
Name        : wikipedia
Description : A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
Arguments   : {'query': {'description': 'query to look up on wikipedia', 'title': 'Query', 'type': 'string'}}


In [7]:
import json
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.tools import PythonREPLTool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_ollama import ChatOllama


# 1. DuckDuckGo
search_tool = DuckDuckGoSearchRun()

# 2. Python REPL
python_tool = PythonREPLTool()

# 3. Wikipedia
api_wrapper = WikipediaAPIWrapper(top_k_results=1,
                                  doc_content_chars_max=500)

wikipedia_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


# Toolkit
tools = [search_tool, python_tool, wikipedia_tool]

d:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
print(tools)

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')), PythonREPLTool(python_repl=PythonREPL(globals={'__name__': 'langchain_experimental.tools.python.tool', '__doc__': 'A tool for running python code in a REPL.', '__package__': 'langchain_experimental.tools.python', '__loader__': <_frozen_importlib_external.SourceFileLoader object at 0x000001AA3E686BD0>, '__spec__': ModuleSpec(name='langchain_experimental.tools.python.tool', loader=<_frozen_importlib_external.SourceFileLoader object at 0x000001AA3E686BD0>, origin='d:\\Dev_Workspace\\LangChain\\langchainenv\\Lib\\site-packages\\langchain_experimental\\tools\\python\\tool.py'), '__file__': 'd:\\Dev_Workspace\\LangChain\\langchainenv\\Lib\\site-packages\\langchain_experimental\\tools\\python\\tool.py', '__cached__': 'd:\\Dev_Workspace\\LangChain\\langchainenv\\Lib\\site-packages\\langchain_experimental\\tools\\python\\__pycache__\\tool.c

In [9]:
# Initialize the language model
llm = ChatOllama(model="qwen3:8b")
llm_with_tools = llm.bind_tools(tools)

In [10]:

for tool in llm_with_tools.kwargs["tools"]:
    function = tool["function"]

    print("=" * 60)
    print(f"🔧 Tool: {function['name']}")
    print("=" * 60)
    print(f"Description:\n{function['description']}")
    print("\nParameters:")

    print(json.dumps(
        function["parameters"],
        indent=4
    ))

    print()

🔧 Tool: duckduckgo_search
Description:
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.

Parameters:
{
    "properties": {
        "query": {
            "description": "search query to look up",
            "type": "string"
        }
    },
    "required": [
        "query"
    ],
    "type": "object"
}

🔧 Tool: Python_REPL
Description:
A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.

Parameters:
{
    "properties": {
        "query": {
            "type": "string"
        }
    },
    "required": [
        "query"
    ],
    "type": "object"
}

🔧 Tool: wikipedia
Description:
A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.

Paramete

In [11]:
response = llm_with_tools.invoke("Who is the current president of the United States?")
print(response.tool_calls)

[{'name': 'duckduckgo_search', 'args': {'query': 'current president of the United States'}, 'id': '7e48edb9-ee47-4dd8-b04d-a16119741eed', 'type': 'tool_call'}]


In [12]:
# Execute the requested tool
tool_call = response.tool_calls[0]
tool_name = tool_call["name"]
tool_args = tool_call["args"]

In [13]:
print("LLM requested:")
print(response.tool_calls)

# Execute the requested tool
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

for tool in tools:
    if tool.name == tool_name:
        tool_result = tool.invoke(tool_args)
        break

print("\nTool result:")
print(tool_result)

LLM requested:
[{'name': 'duckduckgo_search', 'args': {'query': 'current president of the United States'}, 'id': '7e48edb9-ee47-4dd8-b04d-a16119741eed', 'type': 'tool_call'}]

Tool result:
List of all presidents of the United States. No.[a]. Portrait.↑ The 1796 presidential election was the first contested American presidential election and the only one in which a president and vice president were elected from opposing political parties. Learn about the duties of president, vice president, and first lady of the United States. Find out how to contact and learn more about current and past leaders. Under President Donald Trump’s second administration, the United States has surged into a new era of prosperity, marked by record-setting economic growth and trillions in new private-sector investments fueled by tax reforms, deregulation, and a renewed focus on American innovation. As the head of the government of the United States, the president is arguably the most powerful government officia

In [14]:
response = llm_with_tools.invoke("print(sum(range(10)))")
print(response.tool_calls)

[{'name': 'Python_REPL', 'args': {'query': 'print(sum(range(10)))'}, 'id': 'e13ef612-63cf-48aa-9f2b-ad60dd0bbaf5', 'type': 'tool_call'}]


In [15]:
print("LLM requested:")
print(response.tool_calls)

# Execute the requested tool
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

for tool in tools:
    if tool.name == tool_name:
        tool_result = tool.invoke(tool_args)
        break

print("\nTool result:")
print(tool_result)

LLM requested:
[{'name': 'Python_REPL', 'args': {'query': 'print(sum(range(10)))'}, 'id': 'e13ef612-63cf-48aa-9f2b-ad60dd0bbaf5', 'type': 'tool_call'}]

Tool result:
45

